In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime
from nba_api.stats.endpoints import leaguedashteamstats

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict

pd.set_option('display.max_columns', None)

In [2]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()


print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260321_113039.csv
DFS file: NBA_DFS_20260321_113134.csv
DFS latest pull: 2026-03-21 11:31:34
US latest pull: 2026-03-21 11:30:39


In [3]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260321_113133.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Oklahoma City Thunder,2026-03-21 21:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Charlotte Hornets,Memphis Grizzlies,2026-03-21 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,New Orleans Pelicans,Cleveland Cavaliers,2026-03-21 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Orlando Magic,Los Angeles Lakers,2026-03-21 23:11:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Atlanta Hawks,Golden State Warriors,2026-03-22 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [4]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE
86,2025-26,203991,Clint Capela,Clint,1610612745,HOU,Houston Rockets,22501018,2026-03-20T00:00:00,HOU vs. ATL,W,13.791667,1,2,0.500,0,0,0.0,1,4,0.25,2,3,5,2,1,1,0,1,0,2,3,8,14.0,0,0,12.0,1,13:48,1,111.5,112.9,112.9,89.8,87.1,87.1,21.7,25.8,25.8,0.182,2.0,28.6,0.125,0.214,0.167,14.3,14.8,0.500,0.399,0.135,0.127,106.99,107.89,89.91,107.89,0.084,31,1.0,2.0,42,83,0.506,14,30,0.467,19,28,0.679,12,39,51,33,20.0,11,5,9,13,21,117,22.0,113.2,117.0,93.6,93.1,19.6,23.9,0.786,1.65,22.1,0.348,0.755,0.566,0.200,0.590,0.614,102.4,101.0,84.17,100,0.623,1610612737,ATL,Atlanta Hawks,36,85,0.424,9,35,0.257,14,17,0.824,9,28,37,22,18.0,13,9,5,21,13,95,-22.0,93.6,93.1,113.2,117.0,-19.6,-23.9,0.611,1.22,16.4,0.245,0.652,0.434,0.176,0.476,0.514,102.4,101.0,84.17,102,0.377
87,2025-26,1642864,Hugo González,Hugo,1610612738,BOS,Boston Celtics,22501019,2026-03-20T00:00:00,BOS @ MEM,W,14.561667,2,2,1.000,1,1,1.0,0,0,0.00,0,5,5,1,0,0,0,0,3,0,5,7,12.5,0,0,12.0,1,14:34,1,142.4,143.3,143.3,121.5,120.0,120.0,20.9,23.3,23.3,0.067,0.0,33.3,0.000,0.333,0.208,0.0,0.0,1.250,1.250,0.061,0.062,98.63,98.89,82.41,98.89,0.110,30,2.0,2.0,40,89,0.449,11,42,0.262,26,30,0.867,18,39,57,19,13.0,4,2,3,20,23,117,5.0,120.4,120.6,112.6,115.5,7.8,5.2,0.475,1.46,13.9,0.404,0.784,0.592,0.134,0.511,0.572,98.3,97.0,80.83,97,0.526,1610612763,MEM,Memphis Grizzlies,42,90,0.467,14,43,0.326,14,17,0.824,7,28,35,24,9.0,7,3,2,23,20,112,-5.0,112.6,115.5,120.4,120.6,-7.8,-5.2,0.571,2.67,18.5,0.216,0.596,0.408,0.093,0.544,0.574,98.3,97.0,80.83,97,0.474
88,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,22501019,2026-03-20T00:00:00,BOS @ MEM,W,23.216667,1,2,0.500,0,1,0.0,0,0,0.00,2,2,4,3,0,1,0,0,1,2,2,7,14.3,0,0,11.0,1,23:13,1,132.2,130.6,130.6,115.4,116.3,116.3,16.8,14.3,14.3,0.125,0.0,60.0,0.091,0.080,0.085,0.0,0.0,0.500,0.500,0.035,0.036,101.10,101.31,84.42,101.31,0.067,49,1.0,2.0,40,89,0.449,11,42,0.262,26,30,0.867,18,39,57,19,13.0,4,2,3,20,23,117,5.0,120.4,120.6,112.6,115.5,7.8,5.2,0.475,1.46,13.9,0.404,0.784,0.592,0.134,0.511,0.572,98.3,97.0,80.83,97,0.526,1610612763,MEM,Memphis Grizzlies,42,90,0.467,14,43,0.326,14,17,0.824,7,28,35,24,9.0,7,3,2,23,20,112,-5.0,112.6,115.5,120.4,120.6,-7.8,-5.2,0.571,2.67,18.5,0.216,0.596,0.408,0.093,0.544,0.574,98.3,97.0,80.83,97,0.474
90,2025-26,1630700,Dyson Daniels,Dyson,1610612737,ATL,Atlanta Hawks,22501018,2026-03-20T00:00:00,ATL @ HOU,L,25.166667,1,5,0.200,0,0,0.0,1,2,0.50,0,3,3,

In [ ]:
array(['player_points', 'player_rebounds', 'player_assists',
       'player_threes', 'player_blocks', 'player_steals',
       'player_field_goals', 'player_frees_made', 'player_frees_attempts',
       'player_points_rebounds_assists', 'player_points_rebounds',
       'player_points_assists', 'player_rebounds_assists',
       'player_turnovers', 'player_blocks_steals'], dtype=object)

In [11]:
# Generalized best bets for all PrizePicks categories
# (Over/Under best odds from US books + EV using rolling stat distribution)

prop_categories = [
    'player_points',
    'player_rebounds',
    'player_assists',
    'player_turnovers',
    'player_frees_attempts',
    'player_threes',
    'player_blocks',
    'player_steals',
    'player_blocks_steals',
    'player_points_rebounds_assists',
    'player_points_rebounds',
    'player_points_assists',
    'player_rebounds_assists',
]

cat_to_stat_cols = {
    'player_points': ['PTS'],
    'player_rebounds': ['REB'],
    'player_assists': ['AST'],
    'player_turnovers': ['TOV'],
    'player_frees_attempts': ['FTA'],
    'player_threes': ['FG3M'],
    'player_blocks': ['BLK'],
    'player_steals': ['STL'],
    'player_blocks_steals': ['BLK', 'STL'],
    'player_points_rebounds_assists': ['PTS', 'REB', 'AST'],
    'player_points_rebounds': ['PTS', 'REB'],
    'player_points_assists': ['PTS', 'AST'],
    'player_rebounds_assists': ['REB', 'AST'],
}


def implied_prob(american_odds: float) -> float:
    if american_odds > 0:
        return round(100 / (american_odds + 100), 3)
    return round(abs(american_odds) / (abs(american_odds) + 100), 3)


def calc_ev(prob: float, american_odds: float) -> float:
    decimal = (american_odds / 100 + 1) if american_odds > 0 else (100 / abs(american_odds) + 1)
    return round(((prob * (decimal - 1)) - (1 - prob)) * 100, 2)


def _norm_matchup_opp(name: pd.Series) -> pd.Series:
    """Map alternate book/log spellings to one key (e.g. LA Clippers vs Los Angeles Clippers)."""
    s = name.astype(str).str.strip().str.lower()
    return s.replace({'la clippers': 'los angeles clippers'})


# --- Parse game odds: consensus spread & total per team ---
if 'game_odds_df' not in globals():
    game_rows = []

    for game in team_dds.to_dict('records'):
        home = game['home_team']
        away = game['away_team']
        commence = game['commence_time']
        bookmakers = game['bookmakers']

        spreads_home, spreads_away, totals = [], [], []

        for bk in bookmakers:
            for market in bk['markets']:
                if market['market_key'] == 'spreads':
                    for outcome in market['outcomes']:
                        if outcome['name'] == home:
                            spreads_home.append(outcome['point'])
                        elif outcome['name'] == away:
                            spreads_away.append(outcome['point'])
                elif market['market_key'] == 'totals':
                    for outcome in market['outcomes']:
                        if outcome['name'] == 'Over':  # one side is enough
                            totals.append(outcome['point'])

        # Consensus = median across bookmakers; count = how many books reported
        consensus_total = round(np.median(totals), 1) if totals else None
        consensus_spread_home = round(np.median(spreads_home), 1) if spreads_home else None
        consensus_spread_away = round(np.median(spreads_away), 1) if spreads_away else None
        n_books = len(bookmakers)

        game_rows.append({
            'TEAM': home,
            'OPPONENT': away,
            'TEAM_SPREAD': consensus_spread_home,
            'GAME_TOTAL': consensus_total,
            'N_BOOKS': n_books,
            'COMMENCE_TIME': commence,
            'HOME_AWAY': 'HOME',
        })
        game_rows.append({
            'TEAM': away,
            'OPPONENT': home,
            'TEAM_SPREAD': consensus_spread_away,
            'GAME_TOTAL': consensus_total,
            'N_BOOKS': n_books,
            'COMMENCE_TIME': commence,
            'HOME_AWAY': 'AWAY',
        })

    game_odds_df = pd.DataFrame(game_rows)


# Base player log features
# base_df = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')

outputs = []

for category in prop_categories:
    stat_cols = cat_to_stat_cols[category]

    # --- Filter to players with Underdog lines for THIS category ---
    updated_names = []
    cat_underdog_names = lines_dfs[
        (lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == category)
    ]['NAME'].unique()

    for name in cat_underdog_names:
        updated_names.append(nameDict.get(name, name))

    df = base_df[base_df['PLAYER_NAME'].isin(updated_names)].copy()

    if df.empty:
        continue

    # Build the stat we are analyzing for this category (single stat or sum for combos)
    if len(stat_cols) == 1:
        df['STAT_VALUE'] = df[stat_cols[0]]
    else:
        df['STAT_VALUE'] = df[stat_cols].sum(axis=1)

    # --- Rolling stats (last 5 / last 10) for THIS stat ---
    df['AVG_MIN_L5'] = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).mean().round(2))
    df['STD_MIN_L5'] = df.groupby('PLAYER_ID')['MIN'].transform(lambda x: x.rolling(5).std().round(2))
    df['AVG_USG_L5'] = df.groupby('PLAYER_ID')['USG_PCT'].transform(lambda x: x.rolling(5).mean().round(2))
    df['STD_USG_L5'] = df.groupby('PLAYER_ID')['USG_PCT'].transform(lambda x: x.rolling(5).std().round(2))

    df['AVG_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).mean().round(2))
    df['STD_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).std().round(2))
    df['MED_STAT_L5'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(5).median().round(2))
    df['STD_STAT_L10'] = df.groupby('PLAYER_ID')['STAT_VALUE'].transform(lambda x: x.rolling(10).std().round(2))

    df['MIN_CONSISTENCY'] = (df['AVG_MIN_L5'] / df['STD_MIN_L5']).round(2)

    # --- Most recent row per player ---
    latest = df.groupby('PLAYER_ID').last().reset_index()

    # --- PrizePicks lines for THIS category ---
    prop_lines = (
        lines_dfs[(lines_dfs['BOOKMAKER'] == 'Underdog') & (lines_dfs['CATEGORY'] == category)]
        [['NAME', 'LINE', 'ODDS', 'COMMENCE_TIME']]
        .rename(columns={'NAME': 'PLAYER_NAME'})
        .copy()
    )

    if prop_lines.empty:
        continue

    prop_lines['LINE'] = prop_lines['LINE'].astype(float)

    # Keep one line per player for this category (consistent with your existing logic)
    prop_lines = prop_lines.drop_duplicates('PLAYER_NAME')

    merged = latest.merge(prop_lines, on='PLAYER_NAME', how='inner')
    merged['CATEGORY'] = category

    # --- Real book odds (best available per side) ---
    real_odds = us_df[us_df['CATEGORY'] == category].rename(columns={'NAME': 'PLAYER_NAME'}).copy()

    # Ensure numeric odds/lines
    real_odds['LINE'] = real_odds['LINE'].astype(float)

    for side, col in [('Over', 'ODDS_OVER'), ('Under', 'ODDS_UNDER')]:
        best = (
            real_odds[real_odds['OVER/UNDER'] == side]
            .groupby(['PLAYER_NAME', 'LINE'])['ODDS'].max()
            .reset_index()
            .rename(columns={'ODDS': col})
        )
        merged = merged.merge(best, on=['PLAYER_NAME', 'LINE'], how='left')

    merged['ODDS_OVER'] = merged['ODDS_OVER'].fillna(-137).astype(int)
    merged['ODDS_UNDER'] = merged['ODDS_UNDER'].fillna(-137).astype(int)

    # --- Merge game-level spread & total ---
    merged = merged.merge(
        game_odds_df[['TEAM', 'OPPONENT', 'TEAM_SPREAD', 'GAME_TOTAL', 'N_BOOKS', 'HOME_AWAY']],
        left_on='TEAM_NAME',
        right_on='TEAM',
        how='left'
    ).drop(columns='TEAM')

    # --- Season avg of this category's stat vs upcoming opponent (STAT_VALUE matches prop_categories) ---
    if 'OPP_OPP_NAME_base' in df.columns:
        _hist = df.dropna(subset=['OPP_OPP_NAME_base']).copy()
        _hist['_OPP_NORM'] = _norm_matchup_opp(_hist['OPP_OPP_NAME_base'])
        merged['_OPP_NORM'] = _norm_matchup_opp(merged['OPPONENT'])
        matchup_agg = (
            _hist.groupby(['PLAYER_NAME', '_OPP_NORM'], as_index=False)
            .agg(AVG_STAT_VS_MATCHUP=('STAT_VALUE', 'mean'), MATCHUP_GAMES=('STAT_VALUE', 'count'))
        )
        matchup_agg['AVG_STAT_VS_MATCHUP'] = matchup_agg['AVG_STAT_VS_MATCHUP'].round(2)
        merged = merged.merge(matchup_agg, on=['PLAYER_NAME', '_OPP_NORM'], how='left')
        merged = merged.drop(columns=['_OPP_NORM'])
        merged['MATCHUP_EDGE'] = (merged['AVG_STAT_VS_MATCHUP'] - merged['LINE']).round(2)
    else:
        merged['AVG_STAT_VS_MATCHUP'] = np.nan
        merged['MATCHUP_GAMES'] = np.nan
        merged['MATCHUP_EDGE'] = np.nan

    # --- Implied probability from book odds ---
    merged['IMP_PROB_OVER'] = merged['ODDS_OVER'].apply(implied_prob)
    merged['IMP_PROB_UNDER'] = merged['ODDS_UNDER'].apply(implied_prob)

    # --- Core metrics ---
    merged['EDGE'] = (merged['AVG_STAT_L5'] - merged['LINE']).round(2)
    merged['MED_EDGE'] = (merged['MED_STAT_L5'] - merged['LINE']).round(2)
    merged['Z_SCORE'] = ((merged['LINE'] - merged['AVG_STAT_L5']) / merged['STD_STAT_L10']).round(3)

    merged['PROB_OVER'] = (1 - stats.norm.cdf(merged['Z_SCORE'])).round(3)
    merged['PROB_UNDER'] = stats.norm.cdf(merged['Z_SCORE']).round(3)

    # --- Game-context features ---
    merged['TOTAL_BOOST'] = ((merged['GAME_TOTAL'] - 220) / 10).round(3)  # ~0 at league avg
    merged['IS_UNDERDOG'] = (merged['TEAM_SPREAD'] > 0).astype(int)

    # --- Cover rate (last 10-ish windows) ---
    cover_df = df.merge(merged[['PLAYER_NAME', 'LINE']], on='PLAYER_NAME', how='inner')

    cover_windows = {'L5': 5, 'L10': 10, 'L15': 15}

    cover = cover_df.groupby('PLAYER_NAME').apply(
        lambda g: pd.Series({
            'OVER_RATE_L5': (g['STAT_VALUE'].tail(5) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_L10': (g['STAT_VALUE'].tail(10) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_L15': (g['STAT_VALUE'].tail(15) > g['LINE'].iloc[0]).mean().round(2),
            'OVER_RATE_SEASON': (g['STAT_VALUE'] > g['LINE'].iloc[0]).mean().round(2),
        })
    ).reset_index()

    merged = merged.merge(cover, on='PLAYER_NAME', how='left')

    # --- EV % ---
    merged['EV_OVER'] = merged.apply(lambda r: calc_ev(r['PROB_OVER'], r['ODDS_OVER']), axis=1)
    merged['EV_UNDER'] = merged.apply(lambda r: calc_ev(r['PROB_UNDER'], r['ODDS_UNDER']), axis=1)

    # --- Filters + scoring ---
    merged = merged[(merged['AVG_MIN_L5'] >= 20) & (merged['STD_MIN_L5'] <= 8)].copy()

    merged['CONFIDENCE'] = (
        (merged['EDGE'] / merged['STD_STAT_L5'])
        + merged['OVER_RATE_L10']
        + merged['MIN_CONSISTENCY'] * 0.1
        + merged['TOTAL_BOOST'] * 0.15
    ).round(2)

    # Simple, stat-agnostic bet rule: value by EV and probability
    merged['BET_FLAG'] = (
        (merged['PROB_OVER'] >= 0.60)
        & (merged['EV_OVER'] > 0)
    )

    output_cat = merged[[
        'PLAYER_NAME',
        'TEAM_NAME',
        'OPPONENT',
        'HOME_AWAY',
        'TEAM_SPREAD',
        'GAME_TOTAL',
        'CATEGORY',
        'LINE',
        'ODDS_OVER',
        'ODDS_UNDER',
        'IMP_PROB_OVER',
        'IMP_PROB_UNDER',
        'AVG_STAT_L5',
        'MED_STAT_L5',
        'AVG_STAT_VS_MATCHUP',
        'MATCHUP_GAMES',
        'MATCHUP_EDGE',
        'STD_STAT_L5',
        'EDGE',
        'MED_EDGE',
        'Z_SCORE',
        'PROB_OVER',
        'PROB_UNDER',
        'EV_OVER',
        'EV_UNDER',
        'OVER_RATE_L5',
        'OVER_RATE_L10',
        'OVER_RATE_L15',
        'OVER_RATE_SEASON',
        'AVG_MIN_L5',
        'STD_MIN_L5',
        'AVG_USG_L5',
        'STD_USG_L5',
        'MIN_CONSISTENCY',
        'TOTAL_BOOST',
        'IS_UNDERDOG',
        'CONFIDENCE',
        'BET_FLAG',
        'COMMENCE_TIME',
    ]].sort_values('EV_OVER', ascending=False)

    outputs.append(output_cat)


output_all = pd.concat(outputs, ignore_index=True) if outputs else pd.DataFrame()

tier1_all = output_all[output_all['BET_FLAG']] if not output_all.empty else output_all

print('Total bets across categories:', len(output_all))
print('Tier 1 bets:', len(tier1_all))

if not output_all.empty:
    display(tier1_all.head(20))

# --- Opponent defense / pace (LeagueDashTeamStats) ---
if not output_all.empty:
    league_df = leaguedashteamstats.LeagueDashTeamStats(
        league_id_nullable='00',
        per_mode_detailed='PerGame',
        measure_type_detailed_defense='Advanced',
    ).get_data_frames()[0]

    opp_stats = (
        league_df[['TEAM_NAME', 'DEF_RATING', 'DEF_RATING_RANK', 'PACE', 'PACE_RANK']]
        .copy()
        .rename(columns={
            'TEAM_NAME': 'OPPONENT',
            'DEF_RATING': 'OPP_DEF_RATING',
            'DEF_RATING_RANK': 'OPP_RANK_DEF_RATING',
            'PACE': 'OPP_PACE',
            'PACE_RANK': 'OPP_PACE_RANK',
        })
    )

    # NBA team stats use short city names (e.g. LA Clippers) while props may use full names
    _opp_lookup_map = {'Los Angeles Clippers': 'LA Clippers'}
    _output = output_all.assign(
        _OPP_MATCH_KEY=output_all['OPPONENT'].replace(_opp_lookup_map)
    )
    _opp = opp_stats.rename(columns={'OPPONENT': '_OPP_MATCH_KEY'})
    final = _output.merge(_opp, on='_OPP_MATCH_KEY', how='left').drop(columns='_OPP_MATCH_KEY')

    final = final[[
        'PLAYER_NAME', 'LINE', 'CATEGORY', 'OPPONENT',
        'TEAM_SPREAD', 'GAME_TOTAL', 'OPP_DEF_RATING', 'OPP_RANK_DEF_RATING', 'OPP_PACE', 'OPP_PACE_RANK',
        'ODDS_OVER', 'ODDS_UNDER',
        'IMP_PROB_OVER', 'IMP_PROB_UNDER',
        'AVG_STAT_L5', 'MED_STAT_L5', 'STD_STAT_L5', 'EDGE', 'MED_EDGE', 'Z_SCORE',
        'PROB_OVER', 'PROB_UNDER', 'EV_OVER', 'EV_UNDER',
        'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON',
        'AVG_MIN_L5', 'STD_MIN_L5', 'AVG_USG_L5', 'STD_USG_L5', 'MIN_CONSISTENCY', 'IS_UNDERDOG', 'AVG_STAT_VS_MATCHUP', 'MATCHUP_GAMES',
    ]].sort_values('EV_OVER', ascending=False)

    display(final.head())
else:
    final = pd.DataFrame()

/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3960427895.py:235: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3960427895.py:235: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3

Total bets across categories: 284
Tier 1 bets: 88


/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3960427895.py:235: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3960427895.py:235: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/var/folders/9q/5_554qsx5z70w9d_vkmvjg0h0000gn/T/ipykernel_74878/3

,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L5,MED_STAT_L5,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES,MATCHUP_EDGE,STD_STAT_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,AVG_USG_L5,STD_USG_L5,MIN_CONSISTENCY,TOTAL_BOOST,IS_UNDERDOG,CONFIDENCE,BET_FLAG,COMMENCE_TIME
0,Daniel Gafford,Dallas Mavericks,Los Angeles Clippers,HOME,8.0,234.0,player_points,9.5,-106,-114,0.515,0.533,18.6,21.0,7.25,4.0,-2.25,5.27,9.1,11.5,-1.504,0.934,0.066,81.51,-87.61,1.0,0.7,0.53,0.56,24.65,1.90,0.20,0.04,12.97,1.40,1,3.93,True,2026-03-22
1,Bam Adebayo,Miami Heat,Houston Rockets,AWAY,3.0,228.0,player_points,21.5,100,-114,0.500,0.533,35.2,24.0,17.33,3.0,-4.17,26.90,13.7,2.5,-0.723,0.765,0.235,53.00,-55.89,0.6,0.7,0.60,0.34,37.44,3.11,0.33,0.13,12.04,0.80,1,2.53,True,2026-03-22
2,Devin Booker,Phoenix Suns,Milwaukee Bucks,HOME,-11.5,218.0,player_points,27.5,-110,-107,0.524,0.517,34.0,34.0,28.33,3.0,0.83,8.22,6.5,6.5,-0.839,0.799,0.201,52.54,-61.11,0.8,0.6,0.47,0.41,35.02,1.39,0.39,0.04,25.19,-0.20,0,3.88,True,2026-03-22
3,Nickeil Alexander-Walker,Atlanta Hawks,Golden State Warriors,HOME,-10.5,229.5,player_points,18.5,-115,-110,0.535,0.524,24.4,21.0,11.00,5.0,-7.50,9.40,5.9,2.5,-0.781,0.783,0.217,46.39,-58.57,0.8,0.7,0.60,0.31,32.97,4.09,0.22,0.05,8.06,0.95,0,2.28,True,2026-03-22
4,Ryan Rollins,Milwaukee Bucks,Phoenix Suns,AWAY,11.5,218.0,player_points,15.5,-105,-105,0.512,0.512,18.4,19.0,15.00,3.0,-0.50,2.88,2.9,3.5,-0.616,0.731,0.269,42.72,-47.48,0.8,0.4,0.47,0.32,31.45,5.13,0.20,0.03,6.13,-0.20,1,1.99,True,2026-03-22
5,Evan Mobley,Cleveland Cavaliers,New Orleans Pelicans,AWAY,-4.5,236.5,player_points,20.5,-107,-105,0.517,0.512,23.6,26.0,12.00,2.0,-8.50,5.22,3.1,5.5,-0.635,0.737,0.263,42.58,-48.65,0.6,0.5,0.33,0.39,32.05,5.34,0.26,0.07,6.00,1.65,0,1.94,True,2026-03-21
6,Pelle Larsson,Miami Heat,Houston Rockets,AWAY,3.0,228.0,player_points,11.5,-105,-106,0.512,0.515,15.6,14.0,20.00,1.0,8.50,7.23,4.1,2.5,-0.609,0.729,0.271,42.33,-47.33,0.6,0.5,0.40,0.27,31.67,5.49,0.17,0.04,5.77,0.80,1,1.76,True,2026-03-22
7,Davion Mitchell,Miami Heat,Houston Rockets,AWAY,3.0,228.0,player_points,9.5,-103,-110,0.507,0.524,11.8,13.0,11.00,2.0,1.50,4.60,2.3,3.5,-0.511,0.695,0.305,36.98,-41.77,0.8,0.6,0.47,0.43,26.99,3.16,0.15,0.03,8.54,0.80,1,2.07,True,2026-03-22
8,Kawhi Leonard,LA Clippers,NaN,NaN,NaN,NaN,player_points,27.5,-125,-102,0.556,0.505,31.6,29.0,NaN,NaN,NaN,7.80,4.1,1.5,-0.657,0.744,0.256,33.92,-49.30,0.8,0.7,0.60,0.37,30.87,3.63,0.33,0.07,8.50,NaN,0,NaN,True,2026-03-22
9,Bobby Portis,Milwaukee Bucks,Phoenix Suns,AWAY,11.5,218.0,player_points,15.5,-112,-103,0.528,0.507,18.4,19.0,NaN,NaN,NaN,6.84,2.9,3.5,-0.537,0.704,0.296,33.26,-41.66,0.6,0.5,0.47,0.36,27.14,3.37,0.28,0.06,8.05,-0.20,1,1.70,True,2026-03-22


,PLAYER_NAME,LINE,CATEGORY,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L5,MED_STAT_L5,STD_STAT_L5,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L5,STD_MIN_L5,AVG_USG_L5,STD_USG_L5,MIN_CONSISTENCY,IS_UNDERDOG,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
121,Pascal Siakam,27.5,player_points_rebounds_assists,San Antonio Spurs,17.5,234.5,110.4,3.0,100.80,12.0,-103,-107,0.507,0.517,36.6,35.0,3.65,9.1,7.5,-1.727,0.958,0.042,88.81,-91.87,1.0,0.9,0.93,0.71,31.61,3.24,0.33,0.03,9.76,1,34.00,3.0
0,Daniel Gafford,9.5,player_points,Los Angeles Clippers,8.0,234.0,115.5,19.0,97.17,28.0,-106,-114,0.515,0.533,18.6,21.0,5.27,9.1,11.5,-1.504,0.934,0.066,81.51,-87.61,1.0,0.7,0.53,0.56,24.65,1.90,0.20,0.04,12.97,1,7.25,4.0
56,Keon Ellis,2.5,player_rebounds,New Orleans Pelicans,-4.5,236.5,117.0,24.0,101.19,11.0,-106,100,0.515,0.500,3.4,3.0,1.14,0.9,0.5,-0.947,0.828,0.172,60.91,-65.60,0.8,0.8,0.73,0.45,27.92,3.21,0.14,0.04,8.70,0,4.00,4.0
109,Coby White,1.5,player_threes,Memphis Grizzlies,-17.5,234.0,116.6,22.0,101.48,10.0,-107,110,0.517,0.476,2.6,2.0,1.52,1.1,0.5,-0.827,0.796,0.204,53.99,-57.16,0.8,0.7,0.67,0.71,21.78,2.77,0.30,0.07,7.86,0,3.50,2.0
57,P.J. Washington,6.5,player_rebounds,Los Angeles Clippers,8.0,234.0,115.5,19.0,97.17,28.0,-102,-118,0.505,0.541,8.6,9.0,2.07,2.1,2.5,-0.750,0.773,0.227,53.08,-58.06,0.8,0.5,0.47,0.60,30.25,2.13,0.20,0.04,14.20,1,7.75,4.0


In [17]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'CATEGORY': 'Prop',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_STAT_L5': 'Avg Stat L5',
    'MED_STAT_L5': 'Med Stat L5',
    'STD_STAT_L5': 'Std Stat L5',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L5': 'Avg Min L5',
    'STD_MIN_L5': 'Std Min L5',
    'AVG_USG_L5': 'Avg USG% L5',
    'STD_USG_L5': 'Std USG% L5',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog',
    'AVG_STAT_VS_MATCHUP': 'Avg Stat vs Matchup',
    'MATCHUP_GAMES': 'Matchup Games',
}

df = final.rename(columns=rename_map)
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['Prop'] = df['Prop'].map(prop_label_map).fillna(df['Prop'])

df = df[[
    'Player',
    'Prop',
    'Line',
    'Opponent',
    'Odds Over',
    'Odds Under',
    'Implied Over',
    'Implied Under',
    'EV Over',
    'EV Under',
    'Avg Stat L5',
    'Std Stat L5',
    'Z Score',
    'Prob Over',
    'Prob Under',
    'OVER L5',
    'OVER L10',
    'OVER L15',
    'Avg Min L5',
    'Std Min L5',
    'Avg USG% L5',
    'Std USG% L5',
    'Avg Stat vs Matchup',
    'Matchup Games',
    'Spread',
    'Total',
    'Opp Def Rating',
    'Opp Def Rank',
    'Opp Pace',
    'Opp Pace Rank',
]].sort_values(by='EV Over', ascending=False)
df.head(10)

,Player,Prop,Line,Opponent,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Stat L5,Std Stat L5,Z Score,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L5,Std Min L5,Avg USG% L5,Std USG% L5,Avg Stat vs Matchup,Matchup Games,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
121,Pascal Siakam,PTS+REB+AST,27.5,San Antonio Spurs,-103,-107,0.507,0.517,88.81,-91.87,36.6,3.65,-1.727,0.958,0.042,1.0,0.9,0.93,31.61,3.24,0.33,0.03,34.00,3.0,17.5,234.5,110.4,3.0,100.80,12.0
0,Daniel Gafford,PTS,9.5,Los Angeles Clippers,-106,-114,0.515,0.533,81.51,-87.61,18.6,5.27,-1.504,0.934,0.066,1.0,0.7,0.53,24.65,1.90,0.20,0.04,7.25,4.0,8.0,234.0,115.5,19.0,97.17,28.0
56,Keon Ellis,REB,2.5,New Orleans Pelicans,-106,100,0.515,0.500,60.91,-65.60,3.4,1.14,-0.947,0.828,0.172,0.8,0.8,0.73,27.92,3.21,0.14,0.04,4.00,4.0,-4.5,236.5,117.0,24.0,101.19,11.0
109,Coby White,3PM,1.5,Memphis Grizzlies,-107,110,0.517,0.476,53.99,-57.16,2.6,1.52,-0.827,0.796,0.204,0.8,0.7,0.67,21.78,2.77,0.30,0.07,3.50,2.0,-17.5,234.0,116.6,22.0,101.48,10.0
57,P.J. Washington,REB,6.5,Los Angeles Clippers,-102,-118,0.505,0.541,53.08,-58.06,8.6,2.07,-0.750,0.773,0.227,0.8,0.5,0.47,30.25,2.13,0.20,0.04,7.75,4.0,8.0,234.0,115.5,19.0,97.17,28.0
1,Bam Adebayo,PTS,21.5,Houston Rockets,100,-114,0.500,0.533,53.00,-55.89,35.2,26.90,-0.723,0.765,0.235,0.6,0.7,0.60,37.44,3.11,0.33,0.13,17.33,3.0,3.0,228.0,112.1,7.0,96.74,29.0
2,Devin Booker,PTS,27.5,Milwaukee Bucks,-110,-107,0.524,0.517,52.54,-61.11,34.0,8.22,-0.839,0.799,0.201,0.8,0.6,0.47,35.02,1.39,0.39,0.04,28.33,3.0,-11.5,218.0,117.6,25.0,98.42,23.0
227,Bam Adebayo,PTS+AST,24.5,Houston Rockets,100,-110,0.500,0.524,52.00,-54.18,38.0,27.14,-0.706,0.760,0.240,0.6,0.7,0.60,37.44,3.11,0.33,0.13,20.00,3.0,3.0,228.0,112.1,7.0,96.74,29.0
122,Evan Mobley,PTS+REB+AST,34.5,New Orleans Pelicans,104,-105,0.490,0.512,51.57,-49.82,38.6,4.83,-0.653,0.743,0.257,0.8,0.5,0.33,32.05,5.34,0.26,0.07,22.00,2.0,-4.5,236.5,117.0,24.0,101.19,11.0
3,Nickeil Alexander-Walker,PTS,18.5,Golden State Warriors,-115,-110,0.535,0.524,46.39,-58.57,24.4,9.40,-0.781,0.783,0.217,0.8,0.7,0.60,32.97,4.09,0.22,0.05,11.00,5.0,-10.5,229.5,113.4,14.0,100.27,16.0


In [18]:
output_path = f'data/props/ev_analysis/underdog.csv'
df.to_csv(output_path, index=False)